In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml import command, load_component
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output
from azureml.core import Workspace, Dataset, Datastore

subscription_id = '813ac1cb-7b8e-45ab-9b3c-8510079c761a'
resource_group = 'MIBLab_Official'
workspace_name = 'miblab-official-ml'

workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "eegfoundationmodeldata")

# Get a handle to the workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

cluster_name = "Standard-FX24mds"
preprocessing_component = load_component(source="./HBNProcessingJob.yml")

In [2]:
input_dir = Input(
	type="uri_folder",
	mode='ro_mount',
	path="azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/HBN/",
)

output_dir = Output(
	type="uri_folder",
	path='azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/HBN-Processed/',
	mode='rw_mount'
)

In [3]:
# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def HBNPreprocessing(input_dir):
	preprocessing = preprocessing_component(input_dir=input_dir)
	return {
        "output_dir": preprocessing.outputs.output_dir
    }

In [4]:
hbnpreprocessing_job = HBNPreprocessing(input_dir=input_dir)
hbnpreprocessing_job.outputs.output_dir = output_dir

In [5]:
# submit job to workspace
hbnpreprocessing_job = ml_client.jobs.create_or_update(
	hbnpreprocessing_job,
    experiment_name="EEGFoundationModel",
)

hbnpreprocessing_job

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading Preprocessing (0.1 MBs): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

Experiment,Name,Type,Status,Details Page
EEGFoundationModel,shy_bag_4xy6g28v1d,pipeline,NotStarted,Link to Azure Machine Learning studio
